In [1]:
from src import systemClasses, baseClasses
import pandas as pd
from tqdm import tqdm
import datetime
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

full_dataset = baseClasses.Data()
full_dataset.data = full_dataset.data.loc[(full_dataset.data.index>datetime.datetime(2024, 11, 29)) & (full_dataset.data.index<datetime.datetime(2024, 12, 5))]

In [2]:
class Calibration():
    def __init__(self, dataset, tini, tend):
        self.dataset = dataset
        self.tini = tini
        self.tend = tend
        self._info()
        self._data()

    def _info(self):
        logFile = baseClasses.LogFile().configurations
        logFile["Start"] = pd.to_datetime(logFile["Start"])
        logFile["End"] = pd.to_datetime(logFile["End"])
        logFile = logFile.loc[(logFile["Start"] <= self.tend) & (logFile["End"] >= self.tini)]
        if len(logFile) == 0:
            self.configurations = None
            print(f"ERROR: No configurations found for the selected time range")
        else:
            self.configurations = logFile
            self.systems = {}
            for index, configurationRow in self.configurations.iterrows():
                mapping = baseClasses.Configuration(configurationRow["Configuration"]).mapping
                hp_systems = mapping.loc[(mapping["BOARD"]<=4)]["SYSTEM"].unique()
                self.systems[configurationRow["Configuration"]] = {"tini": configurationRow["Start"], "tend": configurationRow["End"], "systems": hp_systems}
            if len(self.systems) == 0:
                self.systems = None
                print(f"ERROR: No systems found for the selected time range")
        return self

    def _data(self):
        if self.systems is not None:
            self.sensors = {}
            cnt = 0
            for configuration, info in self.systems.items():
                print(f"Collecting data for configuration '{configuration}' between {info['tini']} and {info['tend']}")
                self.sensors[cnt] = {"configuration": configuration, "tini": max([info["tini"], self.tini]), "tend": min([info["tend"], self.tend]), "sensors": {}}
                for systemName in info["systems"]:
                    if systemName == "EMPTY":
                        continue
                    system = systemClasses.System(self.dataset, systemName)
                    system.muxEqualization()
                    for id, sensor in system.sensors.items():
                        if sensor not in self.sensors[cnt]["sensors"]:
                            self.sensors[cnt]["sensors"][id] = sensor
                        else:
                            self.sensors[cnt]["sensors"][id] = sensor
                            print(f"WARNING: Sensor {sensor} already exists in the configuration {configuration} between {info['tini']} and {info['tend']}")
                cnt += 1
        return self

    def calibrate(self, ref=40525):
        if self.sensors is not None:
            self.calib = {}
            for cnt, sensors in self.sensors.items():
                self.calib[cnt] = {"configuration": sensors["configuration"], "tini": sensors["tini"], "tend": sensors["tend"], "ref":ref, "calib": {}}
                if ref not in sensors["sensors"].keys():
                    print(f"ERROR: Reference sensor {ref} not found in the configuration {sensors['configuration']} between {sensors['tini']} and {sensors['tend']}")
                    continue
                if sensors["sensors"][ref] is None:
                    print(f"ERROR: Reference sensor {ref} is None in the configuration {sensors['configuration']} between {sensors['tini']} and {sensors['tend']}")
                    continue
                ref_sensor = sensors["sensors"][ref]
                for id, sensor in sensors["sensors"].items():
                    if sensor is not None:
                        cc = (
                            sensor.data.loc[(sensor.data.index >= sensors["tini"]) & (sensor.data.index <= sensors["tend"]) & (sensor.data>0) & (sensor.data<90)] -
                            ref_sensor.data.loc[(ref_sensor.data.index >= sensors["tini"]) & (ref_sensor.data.index <= sensors["tend"]) & (ref_sensor.data>0) & (ref_sensor.data<90)]
                            ).mean()
                        err = (
                            sensor.data.loc[(sensor.data.index >= sensors["tini"]) & (sensor.data.index <= sensors["tend"]) & (sensor.data>0) & (sensor.data<90)] -
                            ref_sensor.data.loc[(ref_sensor.data.index >= sensors["tini"]) & (ref_sensor.data.index <= sensors["tend"]) & (ref_sensor.data>0) & (ref_sensor.data<90)]
                            ).std()
                        self.calib[cnt]["calib"][str(id)] = {"cc": 1e3*cc, "cc_err": 1e3*err}
                self.calib[cnt]["calib"] = pd.DataFrame(self.calib[cnt]["calib"]).T
        return self

In [3]:
cal = Calibration(full_dataset, datetime.datetime(2024, 12, 3, 13, 45, 0), datetime.datetime(2024, 12, 3, 14, 15, 0))

Equalizing APA Sensors: 100%|██████████| 16/16 [00:01<00:00, 10.87sensor/s]

ERROR: System (nan) not found


In [4]:
print(cal.configurations)
print(cal.sensors[0])

   Configuration               Start        End
36         empty 2024-12-03 13:00:00 2025-12-12
{'configuration': 'empty', 'tini': datetime.datetime(2024, 12, 3, 13, 45), 'tend': datetime.datetime(2024, 12, 3, 14, 15), 'sensors': {39666: <src.sensorClasses.Sensor object at 0x7fea00632f40>, 39665: <src.sensorClasses.Sensor object at 0x7fea00632f70>, 39664: <src.sensorClasses.Sensor object at 0x7fe9fa961430>, 39667: <src.sensorClasses.Sensor object at 0x7fe9fc06d9d0>, 39661: <src.sensorClasses.Sensor object at 0x7fe9fba80820>, 39660: <src.sensorClasses.Sensor object at 0x7fe9fec1db80>, 39655: <src.sensorClasses.Sensor object at 0x7fe9fec1da00>, 39654: <src.sensorClasses.Sensor object at 0x7fe9fb22f7f0>, 39653: <src.sensorClasses.Sensor object at 0x7fe9fec1aee0>, 39652: <src.sensorClasses.Sensor object at 0x7fe9fab0dc40>, 99999: <src.sensorClasses.Sensor object at 0x7fe9feb61ac0>, 39651: <src.sensorClasses.Sensor object at 0x7fe9ffabbb80>, 39650: <src.sensorClasses.Sensor object at 0x7fe9

In [5]:
cal.calibrate()

In [6]:
import pandas as pd

# Set pandas display options to avoid truncation
pd.set_option('display.max_rows', None)  # Display all rows
pd.set_option('display.max_columns', None)  # Display all columns
pd.set_option('display.width', None)  # Adjust width of the display to avoid truncation

# Now print the dataframes
print(cal.calib[0]["calib"])


                cc    cc_err
39666   -46.252923  4.897267
39665   -25.334160  4.213414
39664    -4.755653  3.879506
39667     1.250517  3.230352
39661   -19.705034  3.062655
39660   -20.471418  3.617951
39655   -41.558643  2.958723
39654   -46.622329  3.352605
39653   -72.538411  2.731964
39652   -40.927282  2.309359
99999   -26.837367  2.589138
39651    -1.448777  2.414703
39650   -26.339826  2.058570
40526   -16.242885  1.803248
40525     0.000000  0.000000
40524   -43.473507  2.047005
39659   -58.455705  2.425037
39658   -45.617642  2.356360
39657   -31.403493  2.902496
39649   -35.708105  2.901023
39648   -51.942744  2.938096
39647   -63.415328  2.212738
39646   -48.879592  2.836024
39644   -50.834180  2.984888
39630   -52.646559  2.684465
39629     5.072632  2.654157
39628   -40.067278  3.295724
39627   -12.435108  3.140790
39626    -6.815532  2.878121
39625   -31.190086  2.689107
39624   -51.849488  2.693651
39623   -29.576739  2.757856
39622   -24.897312  2.482222
39621    -0.45

In [7]:
import pickle
with open('/eos/user/j/jcapotor/RTDdata/calib/all/poff_2024-12-03 13:45:00_2024-12-03 14:15:00.pkl', 'wb') as f:
    pickle.dump({"40525":cal.calib[0]["calib"]}, f)